In [4]:
import pandas as pd
import numpy as np
from scipy.linalg import solve


In [5]:
b = pd.read_csv("nl34_buses.csv")
l = pd.read_csv("nl34_lines.csv")
print(b)

      name  v_nom  type         x          y carrier  unit  location  \
0    NL0 0  380.0   NaN  4.987569  52.160521      AC   NaN       NaN   
1    NL0 1  380.0   NaN  6.251938  51.982219      AC   NaN       NaN   
2   NL0 10  380.0   NaN  4.876408  52.429016      AC   NaN       NaN   
3   NL0 11  380.0   NaN  6.759179  52.248853      AC   NaN       NaN   
4   NL0 12  380.0   NaN  6.474800  53.213385      AC   NaN       NaN   
5   NL0 13  380.0   NaN  5.663999  51.926845      AC   NaN       NaN   
6   NL0 14  380.0   NaN  5.875628  53.189578      AC   NaN       NaN   
7   NL0 15  380.0   NaN  6.189177  52.530256      AC   NaN       NaN   
8   NL0 16  380.0   NaN  4.630595  51.914405      AC   NaN       NaN   
9   NL0 17  380.0   NaN  4.695256  52.373675      AC   NaN       NaN   
10  NL0 18  380.0   NaN  4.307115  52.015733      AC   NaN       NaN   
11  NL0 19  380.0   NaN  6.956673  53.303342      AC   NaN       NaN   
12   NL0 2  380.0   NaN  6.950262  53.123907      AC   NaN      

In [6]:
# clean the data, keeping only columns we need and get rid of empty rows
buses = b[["name"]].copy()
# drop H2 and battery buses, not relevant for our analysis
buses = buses[~buses["name"].str.contains("H2|battery", case=False, na=False)]


# data needed from lines: name, bus0, bus1, reactance x and capacity s_nom
lines = l[["name", "bus0", "bus1", "x", "r", "s_nom", "s_max_pu"]].copy()
# for line capacity: "s_nom" multplied with "s_max_pu" to recover the usable capacity
lines["s_nom"] = lines["s_nom"] * lines["s_max_pu"]
lines = lines.drop(columns=["s_max_pu"])
# drop lines if value is missing or 0 for any of the columns
lines = lines.dropna(subset=["name", "bus0", "bus1", "x", "r", "s_nom"])
lines = lines[(lines["x"] != 0) & (lines["r"] != 0) & (lines["s_nom"] != 0)]
print(lines)

    name    bus0    bus1          x         r        s_nom
0      0   NL0 0  NL0 16   4.511183  0.550144  2377.343656
1      1   NL0 0  NL0 24   2.420979  0.295241  2377.343656
2     10  NL0 13   NL0 9   4.471080  0.545254  2377.343656
3     11  NL0 14  NL0 26   3.619618  0.721519   688.178427
4     12  NL0 14  NL0 28   1.579964  0.314943   688.178427
5     13  NL0 15   NL0 2  10.270335  1.252480  2377.343656
6     14  NL0 15  NL0 25   8.704554  1.735127   688.178427
7     15  NL0 15   NL0 4   1.856714  0.274322  3065.522083
8     16  NL0 16  NL0 21   1.867834  0.227785  2377.343656
9     17  NL0 16  NL0 22   3.394443  0.413956  2377.343656
10    18  NL0 16   NL0 8   1.697147  0.206969  2377.343656
11    19  NL0 17  NL0 21   4.817750  0.587531  2377.343656
12     2   NL0 1  NL0 11   5.607795  0.683877  2377.343656
13    20  NL0 17  NL0 27   1.363718  0.166307  2377.343656
14    21  NL0 18  NL0 21   1.899698  0.231671  2377.343656
15    22  NL0 18  NL0 30   0.834512  0.101770  2377.3436

In [7]:
# reactance-resistance ratio of the lines to check suitability for DC power flow approximation (neglectable ratio > 4)
avg_xr_ratio = (lines["x"] / lines["r"]).mean()
min_xr_ratio = (lines["x"] / lines["r"]).min()
max_xr_ratio = (lines["x"] / lines["r"]).max()
print(f"Average x/r ratio: {avg_xr_ratio:.2f}")
print(f"Minimum x/r ratio: {min_xr_ratio:.2f}")     
print(f"Maximum x/r ratio: {max_xr_ratio:.2f}")

Average x/r ratio: 7.29
Minimum x/r ratio: 5.02
Maximum x/r ratio: 8.20


In [8]:
# indices for buses and lines
buses["bus_idx"] = np.arange(len(buses))
lines["line_idx"] = np.arange(len(lines))
# create a mapping from bus name to bus index
bus_name_to_idx = dict(zip(buses["name"], buses["bus_idx"]))

# map line endpoints to bus indices
lines["bus0_idx"] = lines["bus0"].map(bus_name_to_idx)
lines["bus1_idx"] = lines["bus1"].map(bus_name_to_idx)
print(lines[["name", "bus0", "bus1", "bus0_idx", "bus1_idx"]])

    name    bus0    bus1  bus0_idx  bus1_idx
0      0   NL0 0  NL0 16         0         8
1      1   NL0 0  NL0 24         0        17
2     10  NL0 13   NL0 9         5        33
3     11  NL0 14  NL0 26         6        19
4     12  NL0 14  NL0 28         6        21
5     13  NL0 15   NL0 2         7        12
6     14  NL0 15  NL0 25         7        18
7     15  NL0 15   NL0 4         7        28
8     16  NL0 16  NL0 21         8        14
9     17  NL0 16  NL0 22         8        15
10    18  NL0 16   NL0 8         8        32
11    19  NL0 17  NL0 21         9        14
12     2   NL0 1  NL0 11         1         3
13    20  NL0 17  NL0 27         9        20
14    21  NL0 18  NL0 21        10        14
15    22  NL0 18  NL0 30        10        24
16    23  NL0 19   NL0 2        11        12
17    24  NL0 19  NL0 31        11        25
18    25   NL0 2  NL0 31        12        25
19    26  NL0 20  NL0 24        13        17
20    27  NL0 20   NL0 4        13        28
21    28  

In [9]:
# incidence matrix A, where A[l, bus0] = +1 (from) and A[l, bus1] = -1 (to)
num_buses = len(buses)
num_lines = len(lines)
A = np.zeros((num_lines, num_buses), dtype=int)

for _, row in lines.iterrows():
    bus0 = row["bus0_idx"]
    bus1 = row["bus1_idx"]
    l = row["line_idx"]
    A[l, bus0] = 1
    A[l, bus1] = -1
print("Incidence matrix shape:", A.shape)
print(buses[["bus_idx", "name"]].head(10))
display(pd.DataFrame(A[:24, :20]))
row_sums = A.sum(axis=1)
print("\nUnique row sums of A (should be 0):", np.unique(row_sums))

Incidence matrix shape: (40, 34)
   bus_idx    name
0        0   NL0 0
1        1   NL0 1
2        2  NL0 10
3        3  NL0 11
4        4  NL0 12
5        5  NL0 13
6        6  NL0 14
7        7  NL0 15
8        8  NL0 16
9        9  NL0 17


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,-1
4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,1,0,0,0,0,-1,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,-1,0
7,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,1,0,0,0,0,0,-1,0,0,0,0,0
9,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0



Unique row sums of A (should be 0): [0]


In [10]:
# save the ordering tables for later export
buses[["bus_idx", "name"]].to_csv("nl34_bus_order.csv", index=False)
lines[["line_idx", "name", "bus0", "bus1", "x", "s_nom"]].to_csv("nl34_line_order.csv", index=False)

In [11]:
# define reference bus
ref_bus_name = buses.loc[0, "name"]  # choose the first bus as reference
ref_bus_idx = int(buses.loc[buses["name"] == ref_bus_name, "bus_idx"].iloc[0])  # explanation: get the index of the reference bus
print(f"Reference bus: {ref_bus_name} index({ref_bus_idx})")

Reference bus: NL0 0 index(0)


In [12]:
# line susceptance vector b = 1/x
x = lines["x"].to_numpy(dtype=float)
b = 1 / x

# build diagonal matrix B = diag(b)
B = np.diag(b)

# remove the reference bus column from A to get A_red
A_red = np.delete(A, ref_bus_idx, axis=1)

# compute reduced bus susceptance matrix
B_red = A_red.T @ B @ A_red

print("Reduced bus susceptance matrix B_red shape:", B_red.shape)
print("A_red shape:", A_red.shape)

Reduced bus susceptance matrix B_red shape: (33, 33)
A_red shape: (40, 33)


In [13]:
# solve (instead of invert)
PTDF_red = np.linalg.solve(B_red, (B @ A_red).T).T
print("PTDF_red shape:", PTDF_red.shape)

PTDF_red shape: (40, 33)


In [14]:
# insert zero column back at the slack position
PTDF = np.insert(PTDF_red, ref_bus_idx, 0.0, axis=1)
print("Final PTDF shape:", PTDF.shape)


ptdf_df = pd.DataFrame(
    PTDF,
    index=lines["name"],
    columns=buses["name"]
)

# change column names to Bus, row names to Line for easier reference
ptdf_df.columns.name = "Bus"
ptdf_df.index.name = "Line"

display(ptdf_df.iloc[:20, :20])

ptdf_df.to_csv("nl34_ptdf_matrix.csv")

Final PTDF shape: (40, 34)


Bus,NL0 0,NL0 1,NL0 10,NL0 11,NL0 12,NL0 13,NL0 14,NL0 15,NL0 16,NL0 17,NL0 18,NL0 19,NL0 2,NL0 20,NL0 21,NL0 22,NL0 23,NL0 24,NL0 25,NL0 26
Line,,,,,,,,,,,,,,,,,,,,
0,0.0,-0.389096,-0.236447,-0.325879,-0.250854,-0.445610,-0.245712,-0.256894,-0.720706,-0.394151,-0.650312,-0.252799,-0.253271,-0.211471,-0.638114,-0.682440,-0.666843,-0.149886,-0.252568,-0.242637
1,0.0,-0.610904,-0.763553,-0.674121,-0.749146,-0.554390,-0.754288,-0.743106,-0.279294,-0.605849,-0.349688,-0.747201,-0.746729,-0.788529,-0.361886,-0.317560,-0.333157,-0.850114,-0.747432,-0.757363
10,0.0,0.413607,0.015441,0.311514,0.190350,0.504876,0.182047,0.200105,-0.050855,-0.006149,-0.041218,0.193493,0.194254,0.126748,-0.039548,-0.112653,-0.043481,0.027292,0.193119,0.177080
11,0.0,0.044199,-0.001164,0.051895,0.349605,0.037320,0.595866,0.060292,0.003833,0.000463,0.003107,0.256404,0.233824,-0.009554,0.002981,0.008491,0.003277,-0.002057,0.267485,-0.256818
12,0.0,-0.044199,0.001164,-0.051895,-0.349605,-0.037320,0.404134,-0.060292,-0.003833,-0.000463,-0.003107,-0.256404,-0.233824,0.009554,-0.002981,-0.008491,-0.003277,0.002057,-0.267485,0.256818
13,0.0,0.018350,-0.000483,0.021544,-0.270014,0.015494,-0.167778,0.025030,0.001591,0.000192,0.001290,-0.497113,-0.552131,-0.003966,0.001238,0.003525,0.001361,-0.000854,-0.186267,-0.106619
14,0.0,0.025850,-0.000681,0.030350,-0.380381,0.021826,-0.236356,0.035262,0.002242,0.000271,0.001817,-0.246483,-0.214045,-0.005587,0.001743,0.004966,0.001917,-0.001203,-0.546247,-0.150199
15,0.0,0.542193,-0.014278,0.636591,0.460045,0.457804,0.222087,0.739603,0.047022,0.005685,0.038111,0.550104,0.571922,-0.117195,0.036567,0.104161,0.040203,-0.025235,0.539396,0.079738
16,0.0,0.021403,-0.192983,-0.012544,-0.052832,0.051751,-0.055593,-0.049588,0.199475,-0.349544,-0.474889,-0.051787,-0.051534,-0.073980,-0.591738,0.178926,-0.316522,-0.107050,-0.051911,-0.057244


In [15]:
# sanity check: apply a test transfer and see if the resulting flows make sense
# inject +1 at one bus and withdraw -1 at another bus
inj_bus = buses.loc[9, "name"] if len(buses) > 1 else buses.loc[1, "name"]
wdr_bus = ref_bus_name

inj_idx = int(buses.loc[buses["name"] == inj_bus, "bus_idx"].iloc[0])
wdr_idx = int(buses.loc[buses["name"] == wdr_bus, "bus_idx"].iloc[0])

p_test = np.zeros(len(buses))
p_test[inj_idx] = 1.0
p_test[wdr_idx] = -1.0
print("Sum of test injections (should be 0):", p_test.sum())

f_test = PTDF @ p_test

flow_test_df = pd.DataFrame({
    "line": lines["name"],
    "flow_for_test_transfer": f_test
})

print(f"\nTest transfer: +1 MW at {inj_bus}, -1 MW at {wdr_bus}")
display(flow_test_df.head(10))

Sum of test injections (should be 0): 0.0

Test transfer: +1 MW at NL0 17, -1 MW at NL0 0


,line,flow_for_test_transfer
0,0,-0.394151
1,1,-0.605849
2,10,-0.006149
3,11,0.000463
4,12,-0.000463
5,13,0.000192
6,14,0.000271
7,15,0.005685
8,16,-0.349544
9,17,0.006149


In [13]:
inj_idx = 8
slack_idx = ref_bus_idx

p = np.zeros(len(buses))
p[inj_idx] = 1.0
p[slack_idx] = -1.0

# remove slack entry from injections
p_red = np.delete(p, slack_idx)

# solve reduced DC load flow for voltage angles
theta_red = np.linalg.solve(B_red, p_red)

# reconstruct full angle vector
theta = np.insert(theta_red, slack_idx, 0.0)

# compute line flows from angles
f_dc = np.diag(b) @ A @ theta

# compute line flows from PTDF
f_ptdf = PTDF @ p

# compare
comparison_df = pd.DataFrame({
    "line": lines["name"].values,
    "flow_ptdf": f_ptdf,
    "flow_dc": f_dc,
    "difference": f_ptdf - f_dc
})

print("Max absolute difference:", np.max(np.abs(f_ptdf - f_dc)))
display(comparison_df)

Max absolute difference: 1.979552991012258e-15


,line,flow_ptdf,flow_dc,difference
0,0,-7.207058e-01,-7.207058e-01,2.220446e-16
1,1,-2.792942e-01,-2.792942e-01,0.000000e+00
2,10,-5.085475e-02,-5.085475e-02,-4.857226e-17
3,11,3.833175e-03,3.833175e-03,-4.336809e-19
4,12,-3.833175e-03,-3.833175e-03,9.757820e-17
5,13,1.591356e-03,1.591356e-03,1.214306e-17
6,14,2.241819e-03,2.241819e-03,-2.298509e-17
7,15,4.702158e-02,4.702158e-02,-1.526557e-16
8,16,1.994746e-01,1.994746e-01,-2.220446e-16
9,17,5.085475e-02,5.085475e-02,1.318390e-16
